### Normalizing player names for matching

Thread titles are written by fans, not databases. Matching them to the
official `pfr_player_name` field requires normalizing both sides to a common
form, and deciding what counts as "the name" of a player.

Three problems have to be handled before any matching happens:

- **Suffixes.** "Kenneth Walker III" and "Marvin Harrison Jr." have surnames
  of Walker and Harrison. Naively taking the last token yields "iii" and "jr".
- **Accents and punctuation.** Database spellings and fan spellings diverge on
  characters like apostrophes and accents (Ja'Marr, Muñoz).
- **Case.** Titles are inconsistently capitalized.

Matching is done on the **surname**, since fans rarely write full names —
"Penning was the pick" is far more common than "Trevor Penning was the pick".
The full normalized name is kept alongside it so a full-name match can be
treated as higher-confidence than a surname-only match.

In [ ]:
import re
import unicodedata

SUFFIXES = {"jr", "sr", "ii", "iii", "iv", "v"}

def normalize(text):
    """Lowercase, strip accents, drop punctuation, collapse whitespace."""
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def split_name(full_name):
    """Return (normalized full name, surname), suffixes removed."""
    tokens = [t for t in normalize(full_name).split() if t not in SUFFIXES]
    return " ".join(tokens), (tokens[-1] if tokens else "")

# All six draft classes: 2021-2025 have outcomes, 2026 is the prediction set
picks_2026 = pd.read_csv("../data/processed/draft_2026_picks.csv")
picks = pd.concat([outcomes, picks_2026], ignore_index=True)

picks[["name_norm", "surname"]] = picks["pfr_player_name"].apply(
    lambda n: pd.Series(split_name(n))
)

print("picks:", len(picks))
picks[["season", "team", "round", "pick", "pfr_player_name", "name_norm", "surname"]].head(10)

picks: 1551


,season,team,round,pick,pfr_player_name,name_norm,surname
0,2021,JAX,1,1,Trevor Lawrence,trevor lawrence,lawrence
1,2021,NYJ,1,2,Zach Wilson,zach wilson,wilson
2,2021,SFO,1,3,Trey Lance,trey lance,lance
3,2021,ATL,1,4,Kyle Pitts,kyle pitts,pitts
4,2021,CIN,1,5,Ja'Marr Chase,ja marr chase,chase
5,2021,MIA,1,6,Jaylen Waddle,jaylen waddle,waddle
6,2021,DET,1,7,Penei Sewell,penei sewell,sewell
7,2021,CAR,1,8,Jaycee Horn,jaycee horn,horn
8,2021,DEN,1,9,Patrick Surtain II,patrick surtain,surtain
9,2021,PHI,1,10,DeVonta Smith,devonta smith,smith


In [ ]:
# 1. Players whose suffix was stripped
suffixed = picks[picks["pfr_player_name"].str.contains(
    r"\b(Jr|Sr|II|III|IV|V)\.?$", regex=True, na=False)]
print(f"suffixed names: {len(suffixed)}")
print(suffixed[["pfr_player_name", "surname"]].head(8).to_string(index=False))

# 2. Two players, same team, same year, same surname
dupes = picks[picks.duplicated(["team", "season", "surname"], keep=False)]
print(f"\nsurname collisions within a team-year: {len(dupes)}")
if len(dupes):
    print(dupes.sort_values(["season", "team"])[
        ["season", "team", "round", "pick", "pfr_player_name", "surname"]
    ].to_string(index=False))

# 3. Short surnames — higher false-positive risk
short = picks[picks["surname"].str.len() <= 3]
print(f"\nsurnames of 3 characters or fewer: {short['surname'].nunique()}")
print(sorted(short["surname"].unique()))

suffixed names: 41
     pfr_player_name  surname
  Patrick Surtain II  surtain
     Greg Newsome II  newsome
   Asante Samuel Jr.   samuel
Terrace Marshall Jr. marshall
    Patrick Jones II    jones
   Michael Carter II   carter
    Earnest Brown IV    brown
    Kary Vincent Jr.  vincent

surname collisions within a team-year: 28
 season team  round  pick   pfr_player_name    surname
   2021  DEN      2    35  Javonte Williams   williams
   2021  DEN      6   219     Seth Williams   williams
   2021  LAR      4   117       Bobby Brown      brown
   2021  LAR      5   174  Earnest Brown IV      brown
   2021  NYJ      4   107    Michael Carter     carter
   2021  NYJ      5   154 Michael Carter II     carter
   2022  CHI      3    71   Velus Jones Jr.      jones
   2022  CHI      5   168     Braxton Jones      jones
   2022  GNB      1    22       Quay Walker     walker
   2022  GNB      7   249    Rasheed Walker     walker
   2022  NOR      5   161   D'Marco Jackson    jackson
   2022 

/var/folders/mn/lvfv60bs3q12sv7j_bvffdyc0000gn/T/ipykernel_33696/4203025273.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  suffixed = picks[picks["pfr_player_name"].str.contains(
